# nn-module-subclass — worked example 2: diagnose and fix missing super().__init__()

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `nn-module-subclass`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Assigning an `nn.Parameter` before calling `super().__init__()` raises `AttributeError: cannot assign parameter before Module.__init__() call`, because the registry dict that stores parameters has not been created yet. The fix is always to call `super().__init__()` as the first line of `__init__`.

## Worked solution

We first show the bug: `BrokenBias` assigns `self.bias = nn.Parameter(...)` without calling the base initializer, so constructing it raises an `AttributeError`. We catch and print that error to make the failure mode concrete. Then we write `FixedBias`, identical except `super().__init__()` runs first, which creates the `_parameters` registry so the assignment succeeds and the bias is registered. The `forward` simply adds the bias. We construct the fixed version, confirm it has a registered parameter via `list(.parameters())`, and print both the caught error and the working forward output.

In [ ]:
import torch as t
import torch.nn as nn

class BrokenBias(nn.Module):
    def __init__(self, dim):
        self.bias = nn.Parameter(t.zeros(dim))  # BUG: before super().__init__()
    def forward(self, x):
        return x + self.bias

try:
    BrokenBias(4)
except AttributeError as e:
    print('caught:', str(e)[:60])

class FixedBias(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.bias = nn.Parameter(t.zeros(dim))
    def forward(self, x):
        return x + self.bias

mod = FixedBias(4)
print('num params:', len(list(mod.parameters())))
print(mod(t.ones(4)))